# Train/Test Split - Closed-Set Pairwise Verification

This notebook builds the train/test split for the cantaor embedding project, following the closed-set pairwise verification protocol decided in `01_conteo_cantaores.ipynb`: the same cantaores (with 2+ reliable recordings) appear in both train and test, split at the **recording level** - no single recording appears in both sets, avoiding data leakage.

Input: `data/processed/conteo_cantaores_combinado.csv`, produced by the previous notebook.

In [6]:
import pandas as pd

combined = pd.read_csv('data/processed/conteo_cantaores_combinado.csv', index_col=0)

print(f"Total cantaores: {len(combined)}")
combined[combined['total'] >= 2].sort_values('total', ascending=False)

Total cantaores: 79


,n_c100,nombre_c100,n_tonas,nombre_tonas,total,nombre
antoniomairena,8,Antonio Mairena,6,Antonio Mairena,14,Antonio Mairena
manuelsotosordera,7,"Manuel Soto ""Sordera""",0,NaN,7,"Manuel Soto ""Sordera"""
juantalega,2,Juan Talega,4,Juan Talega,6,Juan Talega
fernandoterremoto,6,Fernando Terremoto,0,NaN,6,Fernando Terremoto
camaron,5,Camarón,0,NaN,5,Camarón
diegoclavel,3,Diego Clavel,2,Diego Clavel,5,Diego Clavel
manuelagujetas,4,Manuel Agujetas,1,Manuel Agujetas,5,Manuel Agujetas
antonionunezchocolate,0,NaN,4,"Antonio Nunez ""Chocolate""",4,"Antonio Nunez ""Chocolate"""
fosforito,4,Fosforito,0,NaN,4,Fosforito
jacintoalmaden,2,Jacinto Almadén,2,Jacinto Almaden,4,Jacinto Almadén


## Loading recording-level data

The combined CSV only has aggregated counts per cantaor. We need individual recording filenames to actually assign specific recordings to train or test. We reload both datasets, reapply the same normalization/filtering as before, and keep only cantaores with 2+ reliable recordings.

In [7]:
import re
import unicodedata

def norm(name):
    if pd.isna(name):
        return ""
    s = str(name)
    s = re.sub(r'\(.*?\)', '', s)
    s = s.replace('"', '')
    s = unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode('ascii')
    s = s.lower().strip()
    s = re.sub(r'\bel\b|\bla\b|\bde\b|\bdel\b', '', s)
    s = re.sub(r'[^a-z]', '', s)
    return s

c100_raw = pd.read_csv('data/raw/cante100_catalogo - cante100_catalogo.csv.csv')
tonas_raw = pd.read_csv('data/raw/tonas_catalogo - tonas_catalogo.csv.csv')

c100_rec = c100_raw.drop_duplicates(subset='hash_audio_1mb', keep='first').copy()
c100_rec['norm_name'] = c100_rec['cantaor_normalizado'].apply(norm)
c100_rec['source'] = 'cante100'
c100_rec['recording_id'] = c100_rec['filename']

tonas_rec = tonas_raw[tonas_raw['confianza_mapeo'].isin(['alta', 'media'])].copy()
tonas_rec['norm_name'] = tonas_rec['cantaor_propuesto'].apply(norm)
tonas_rec['source'] = 'tonas'
tonas_rec['recording_id'] = tonas_rec['archivo_wav']

# Fix the Rafael Romero cross-match manually, same as before
tonas_rec.loc[tonas_rec['norm_name'] == 'rafaelromerogallina', 'norm_name'] = 'rafaelromero'

eligible = set(combined[combined['total'] >= 2].index) - {'rafaelromero_merged'} | {'rafaelromero'}

recordings = pd.concat([
    c100_rec[['norm_name', 'source', 'recording_id']],
    tonas_rec[['norm_name', 'source', 'recording_id']]
])
recordings = recordings[recordings['norm_name'].isin(eligible)].reset_index(drop=True)

print(f"Total individual recordings for eligible cantaores: {len(recordings)}")
recordings.head(10)

Total individual recordings for eligible cantaores: 109


,norm_name,source,recording_id
0,fernandoterremoto,cante100,001_FernandoTerremoto_Fandangos.mp3
1,manuelsotosordera,cante100,002_ManuelSotoSordera_Fandangos.mp3
2,rafaelromero,cante100,003_RafaelRomero_Fandangos.mp3
3,manuelsotosordera,cante100,004_ManuelSotoSordera_Fandangos.mp3
4,fosforito,cante100,005_Fosforito_Fandangos.mp3
5,camaron,cante100,006_Camaron_Fandangos.mp3
6,pacotoronjo,cante100,007_PacoToronjo_Fandangos.mp3
7,pacotoronjo,cante100,008_PacoToronjo_Fandangos.mp3
8,antoniomairena,cante100,011_AntonioMairena_TientosTangos.mp3
9,aurelioselles,cante100,012_AurelioSelles_TientosTangos.mp3


## Building the split: one recording per cantaor held out for test

For each eligible cantaor, we hold out exactly 1 recording for test and keep the rest for train. This guarantees every cantaor has at least 1 training example and 1 test example, which is the minimum needed to form a positive pair at evaluation time (two different recordings of the same person).

Cantaores with more recordings could have more held out for test, but starting with a fixed 1-per-cantaor rule is simpler, reproducible, and keeps as much data as possible for training given the small dataset.

In [8]:
import numpy as np

rng = np.random.default_rng(seed=42)

test_ids = []
train_ids = []

for cantaor, group in recordings.groupby('norm_name'):
    idx = group.index.to_numpy().copy()
    rng.shuffle(idx)
    test_ids.append(idx[0])          # 1 recording held out for test
    train_ids.extend(idx[1:])        # the rest go to train

recordings['split'] = 'train'
recordings.loc[test_ids, 'split'] = 'test'

print(recordings['split'].value_counts())
print()
print(f"Cantaores represented in test: {recordings[recordings['split']=='test']['norm_name'].nunique()}")
print(f"Cantaores represented in train: {recordings[recordings['split']=='train']['norm_name'].nunique()}")

split
train    78
test     31
Name: count, dtype: int64

Cantaores represented in test: 31
Cantaores represented in train: 31


## Saving the split

The recording-level split is saved as the canonical reference for all downstream steps (audio preprocessing, encoder training, evaluation).

In [9]:
recordings.to_csv('data/processed/train_test_split.csv', index=False)
print("Saved to data/processed/train_test_split.csv")
recordings.head()

Saved to data/processed/train_test_split.csv


,norm_name,source,recording_id,split
0,fernandoterremoto,cante100,001_FernandoTerremoto_Fandangos.mp3,train
1,manuelsotosordera,cante100,002_ManuelSotoSordera_Fandangos.mp3,train
2,rafaelromero,cante100,003_RafaelRomero_Fandangos.mp3,test
3,manuelsotosordera,cante100,004_ManuelSotoSordera_Fandangos.mp3,train
4,fosforito,cante100,005_Fosforito_Fandangos.mp3,train


## Note on pair construction for evaluation

With this split, each cantaor has exactly 1 test recording and several train recordings. This means positive pairs for verification are **not** test-vs-test (impossible here, since only 1 recording per cantaor sits in test) - they are formed by comparing each test recording against that same cantaor's recordings in the train set. Negative pairs compare a test recording against train recordings from *different* cantaores.

This is a valid closed-set verification setup (the embedding model still needs to place same-cantaor recordings closer than different-cantaor ones), but it means the "test" recording is compared against material the model *did* see during training for that identity, a slightly easier setting than held-out-pair-vs-held-out-pair verification. This is noted explicitly here so it's clear in the methodology, not discovered as a gap later.

A stricter alternative — holding out 2 recordings per cantaor to allow true test-vs-test pairs — was considered, but would cost training data for cantaores that already have few recordings. Given the dataset size, the current design was chosen deliberately as the better trade-off, and is documented as such.